# Geospatial Data Analysis and Visualization
## Class 10 — Raster Data & Remote Sensing

**Keynote Speaker:** Md Wabidur Rahman, Researcher, Research and Integrated Thought (RIT)
**Class 10 · 23 October 2026**

---

### What this notebook covers

| Part | Topic | Main tool |
|------|------|---------|
| 1 | **NetCDF -> a georeferenced raster** | `rioxarray` |
| 2 | **Raster operations** — reproject, resample | `.rio.reproject()` |
| 3 | **Clip / Mask** — cutting to the shape of Bangladesh | `.rio.clip()` |
| 4 | **Zonal Statistics** — the mean for each district | `clip` + `mean` |
| 5 | **GeoTIFF and COG** — the modern raster formats | `rasterio` |
| 6 | **Downloading data** — ERA5, Sentinel, Landsat | `cdsapi` |
| 7 | **STAC — satellite imagery without downloading** | `pystac-client`, `odc-stac` |
| 8 | **NDVI** — measuring the health of vegetation | band math |

---

### Today's highlight

You have no remote sensing data of your own — that is no problem at all. Today we show —

1. **how to download it** (ERA5, Sentinel-2, Landsat — with real, working code), and
2. **how to use it without downloading at all** — which is the modern approach, called
   **STAC + COG**. Terabytes of imagery stay in the cloud and you read only the
   piece you need. In 2026 this is how professionals work.

In [ ]:
# ════════════════════════════════════════════════════════════════════
#  SETUP — locating and preparing the data  (identical in every notebook)
# ════════════════════════════════════════════════════════════════════
# This cell finds your four datasets by itself:
#   * monthly_mean_data_*.zip / data_0.nc   (ERA5 monthly)
#   * hourly_mean_data_*.zip  / data_0.nc   (ERA5 hourly)
#   * BGD_adm.zip                           (GADM shapefile)
#   * bgd_adm_bbs_*_shp.zip                 (BBS/HDX shapefile)
# If it finds the zips it extracts them into ./data/. If they are already extracted it uses them as they are.

from pathlib import Path
import zipfile, sys, os

DATA = Path.cwd() / "data"
OUT  = Path.cwd() / "outputs"
DATA.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)

# Where to look
SEARCH_ROOTS = [Path.cwd(), Path.cwd()/"data", Path.cwd().parent,
                Path.home()/"Downloads", Path.home()/"Desktop", Path.home()]

def _find(patterns, roots=SEARCH_ROOTS, maxdepth=3):
    """Return the first file that matches the pattern."""
    for root in roots:
        if not root.exists():
            continue
        for pat in patterns:
            try:
                hits = sorted(root.glob(pat)) + sorted(root.glob("*/" + pat))
                if maxdepth >= 3:
                    hits += sorted(root.glob("*/*/" + pat))
                for h in hits:
                    if h.exists():
                        return h
            except Exception:
                pass
    return None

def _ensure(zip_patterns, member_pattern, dest):
    """If member_pattern is not in dest, find the zip and extract it there."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    got = sorted(dest.rglob(member_pattern))
    if got:
        return got[0]
    z = _find(zip_patterns)
    if z is not None:
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        got = sorted(dest.rglob(member_pattern))
        if got:
            print(f"  extracted {z.name}  ->  {dest.name}/")
            return got[0]
    return None

print("Looking for the data ...")
NC_MONTHLY = _ensure(["*monthly_mean*zip"], "*.nc", DATA/"nc_monthly")
NC_HOURLY  = _ensure(["*hourly_mean*zip"],  "*.nc", DATA/"nc_hourly")
SHP_GADM   = _ensure(["*BGD_adm*.zip"], "BGD_adm2.shp", DATA/"shp_adm")
SHP_BBS    = _ensure(["*bgd_adm_bbs*shp*.zip", "*bgd_adm_bbs*.zip"],
                     "bgd_admbnda_adm2*.shp", DATA/"shp_bbs")

# GADM is optional -- only the GADM versus BBS comparison in Class 08 needs
# it (see the README).
status = {"ERA5 monthly (.nc)": NC_MONTHLY, "ERA5 hourly (.nc)": NC_HOURLY,
          "BBS/HDX shapefile": SHP_BBS}
print()
ok = True
for k, v in status.items():
    print(f"  {'✓' if v else '✗'}  {k:24s} {v if v else 'not found'}")
    ok = ok and (v is not None)
print(f"  {'✓' if SHP_GADM else '·'}  {'GADM (optional)':24s} "
      f"{SHP_GADM if SHP_GADM else 'absent -- only the Class 08 comparison is skipped'}")
print()
if ok:
    print("All required data ready")
else:
    print("Some required files were not found — put the zip files next to this notebook and run again.")

In [ ]:
# -- Libraries and shared styling (identical in every notebook) ------------
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib, matplotlib.pyplot as plt
import geopandas as gpd, xarray as xr

# Publication-ready defaults so the figures can go straight into a journal
plt.rcParams.update({
    "figure.dpi": 110,          # for viewing on screen
    "savefig.dpi": 300,         # for saving to a file (journals usually want >=300)
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 10, "axes.grid": True,
    "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "figure.autolayout": False,
})

# Brand colours — the same in every notebook
C_WARM, C_COOL, C_ACC, C_INK = "#B4562F", "#3D7EA6", "#E0A458", "#1B3A5C"

print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("geopandas   ", gpd.__version__)
print("xarray      ", xr.__version__)
print("matplotlib  ", matplotlib.__version__)
#print("rioxarray  ", rioxarray.__version__)
#print("rasterio  ", rasterio.__version__)

In [ ]:
import rioxarray, rasterio
from rasterio.windows import Window
print("rioxarray", rioxarray.__version__, "| rasterio", rasterio.__version__)

BBSDIR = SHP_BBS.parent
country  = gpd.read_file(sorted(BBSDIR.glob("bgd_admbnda_adm0_*.shp"))[0])
division = gpd.read_file(sorted(BBSDIR.glob("bgd_admbnda_adm1_*.shp"))[0])[
    ["ADM1_EN", "geometry"]]
district = gpd.read_file(sorted(BBSDIR.glob("bgd_admbnda_adm2_*.shp"))[0])[
    ["ADM2_EN", "ADM1_EN", "geometry"]]

ds = xr.open_dataset(NC_MONTHLY).squeeze(drop=True)
ds["t_c"] = ds["t"] - 273.15
print(f"Data ready — {len(district)} districts, ERA5 {dict(ds.sizes)}")

---
# Part 1 — Turning a NetCDF into a true RASTER

## 1.1 The problem: xarray does not know this is a map

After opening it with `xr.open_dataset()` we have a **cube of numbers**,
but xarray does not know —

- which **CRS** is it in?
- how much ground does each cell cover?
- which axis is x and which is y?

**`rioxarray`** adds that information — and then xarray can be used as a
true GIS raster, just like QGIS.

In [ ]:
# the 25-year mean temperature — a 2-D raster
t_mean = ds["t_c"].mean("valid_time").rename("t_mean_c")

print("Before adding rioxarray:")
print("  is the CRS known?", t_mean.rio.crs)

# Step 1 — declare the CRS (ERA5 is always WGS 84)
#t_mean = t_mean.rio.write_crs("EPSG:4326")
# Step 2 — declare which axis is x and which is y
#t_mean = t_mean.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude")

print("\nAfter adding rioxarray:")
print("  CRS         :", t_mean.rio.crs)
print("  shape (y, x) :", t_mean.rio.shape)
print("  resolution   :", [round(v, 4) for v in t_mean.rio.resolution()])
print("  bounds       :", [round(v, 3) for v in t_mean.rio.bounds()])
print("\n  Affine transform (the bridge from cell to coordinate):")
print(t_mean.rio.transform())

### What is an affine transform?

It is a rule of six numbers that converts **a cell number into a coordinate on Earth** —

```
| a  b  c |     a = cell size in x      c = x of the top-left corner
| d  e  f |     e = cell size in y      f = y of the top-left corner
| 0  0  1 |     b, d = rotation (usually 0)
```

The formulas: `x = a*col + b*row + c` and `y = d*col + e*row + f`

> **Why is `e` negative?** Because the rows of an image increase **from top to bottom**, while
> latitude increases **from bottom to top**. So latitude **decreases** with each row.
> This is the case in almost every raster.

In [ ]:
# verifying the transform by hand
tr = t_mean.rio.transform()
print("centre of cell (row=0, col=0) -> ", tr * (0.5, 0.5))
print("centre of cell (row=5, col=3) -> ", tr * (3.5, 5.5))

print("\nThe actual coordinates from the data:")
print("  latitude[5]  =", float(t_mean.latitude[5]))
print("  longitude[3] =", float(t_mean.longitude[3]))
print("\nThey match — the transform is correct.")

# how large is one cell?
res_deg = abs(t_mean.rio.resolution()[0])
print(f"\none cell = {res_deg} deg ~ {res_deg*111:.1f} km x "
      f"{res_deg*111*np.cos(np.deg2rad(23.8)):.1f} km (at 23.8N)")
print(f"area of one cell ~ {res_deg*111 * res_deg*111*np.cos(np.deg2rad(23.8)):.0f} km2")

---
# Part 2 — RASTER operations

## 2.1 Reprojection — converting to another CRS

For vectors we had `.to_crs()`. For rasters it is `.rio.reproject()` — but there is one big
difference: **a vector only has its coordinates changed, whereas a raster has its entire grid
rebuilt** (resampling), so the values shift slightly.

In [ ]:
# WGS84 (degrees) -> BTM (metres)

# make sure the CRS is set — this works even if the earlier cell did not run
if t_mean.rio.crs is None:
    t_mean = (t_mean
              .rio.write_crs("EPSG:4326")
              .rio.set_spatial_dims(x_dim="longitude", y_dim="latitude"))

t_btm = t_mean.rio.reproject("EPSG:3106")

print("original (EPSG:4326)")
print(f"  shape {t_mean.rio.shape},  resolution {t_mean.rio.resolution()[0]:.3f} deg")
print("reprojected (EPSG:3106)")
print(f"  shape {t_btm.rio.shape},  resolution {t_btm.rio.resolution()[0]:,.0f} metres")
print(f"\nmean value — before {float(t_mean.mean()):.4f} C, after "
      f"{float(t_btm.where(t_btm != t_btm.rio.nodata).mean()):.4f} °C")
print("-> a small difference is normal, because the grid was rebuilt.")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
t_mean.plot(ax=ax[0], cmap="RdYlBu_r", cbar_kwargs={"label": "°C", "shrink": 0.8})
ax[0].set_title("EPSG:4326 — degrees"); ax[0].set_xlabel("Longitude (°E)")
t_btm.where(t_btm != t_btm.rio.nodata).plot(
    ax=ax[1], cmap="RdYlBu_r", cbar_kwargs={"label": "°C", "shrink": 0.8})
ax[1].set_title("EPSG:3106 — metres (BTM)"); ax[1].set_xlabel("Easting (m)")
plt.tight_layout(); plt.show()

### Resampling methods — which one when

| Method | What it does | When |
|--------|---------|------|
| `nearest` | takes the value of the nearest cell exactly | **categorical** data (land cover, soil type) |
| `bilinear` | a weighted mean of four neighbours | **continuous** data (temperature, elevation) |
| `cubic` | sixteen neighbours, smoother still | continuous data, attractive images |
| `average` | the mean over the area | when going from **fine to coarse** (downsampling) |

In [ ]:
from rasterio.enums import Resampling

# ERA5 is very coarse (0.25 deg ~ 28 km). Let us interpolate onto a finer grid
fine_lat = np.arange(27.0, 19.999, -0.01)
fine_lon = np.arange(87.0, 94.001, 0.01)

t_fine = (t_mean.interp(latitude=fine_lat, longitude=fine_lon, method="linear")
          .rio.write_crs("EPSG:4326")
          .rio.set_spatial_dims(x_dim="longitude", y_dim="latitude"))

print(f"original : {t_mean.rio.shape}  @ 0.25 deg  = {t_mean.size:,} cells")
print(f"fine     : {t_fine.rio.shape}  @ 0.01 deg  = {t_fine.size:,} cells")
print(f"-> {t_fine.size/t_mean.size:.0f} times as many cells")

print("\nAn important warning:")
print("  Interpolation makes the image look smooth, but it creates no new information!")
print("  The native resolution is still 28 km. This must be stated clearly in a journal,")
print("  otherwise a reviewer will see that you are showing false precision.")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
t_mean.plot(ax=ax[0], cmap="RdYlBu_r", cbar_kwargs={"label": "°C", "shrink": .8})
ax[0].set_title(f"Native 0.25° ({t_mean.rio.shape[0]}×{t_mean.rio.shape[1]})")
t_fine.plot(ax=ax[1], cmap="RdYlBu_r", cbar_kwargs={"label": "°C", "shrink": .8})
ax[1].set_title(f"Interpolated 0.01° ({t_fine.rio.shape[0]}×{t_fine.rio.shape[1]})")
for a in ax: a.set_xlabel("Longitude (°E)"); a.set_ylabel("Latitude (°N)")
plt.tight_layout(); plt.show()

---
# Part 3 — Clip / Mask: cutting to the shape of Bangladesh

In Notebook 1 we saw that the ERA5 grid is much larger than Bangladesh — it also covers India,
Myanmar and the Bay of Bengal. For the study we want **only Bangladesh**.

`.rio.clip()` does that. There is one important option —

| | `all_touched=False` (default) | `all_touched=True` |
|---|---|---|
| Which cells it keeps | those whose **centre** is inside the polygon | those that **touch** the polygon |
| Result | fewer cells, gaps along the boundary | more cells, the boundary is covered |
| When | a fine grid | **a coarse grid / small polygons** |

In [ ]:
clip_center = t_mean.rio.clip(country.geometry, country.crs,
                              all_touched=False, drop=True)
clip_touch = t_mean.rio.clip(country.geometry, country.crs,
                             all_touched=True, drop=True)

print(f"original grid        : {t_mean.size:4d} cells")
print(f"clip (centre-based)  : {int(clip_center.notnull().sum()):4d} valid cells")
print(f"clip (all_touched)   : {int(clip_touch.notnull().sum()):4d} valid cells")
print(f"\nMean temperature of Bangladesh:")
print(f"  centre-based  : {float(clip_center.mean()):.3f} C")
print(f"  all_touched   : {float(clip_touch.mean()):.3f} °C")

fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
for a, dat, ttl in [
        (ax[0], t_mean, f"Original grid ({t_mean.size} cells)"),
        (ax[1], clip_center, f"clip, all_touched=False ({int(clip_center.notnull().sum())})"),
        (ax[2], clip_touch, f"clip, all_touched=True ({int(clip_touch.notnull().sum())})")]:
    dat.plot(ax=a, cmap="RdYlBu_r", vmin=24, vmax=27,
             cbar_kwargs={"label": "°C", "shrink": 0.75})
    country.boundary.plot(ax=a, color="k", linewidth=1.1)
    a.set_title(ttl, fontsize=10)
    a.set_xlabel("Longitude (°E)"); a.set_ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
# clipping on the fine grid — it looks much better
fine_clip = t_fine.rio.clip(country.geometry, country.crs, all_touched=True, drop=True)

fig, ax = plt.subplots(figsize=(7.5, 7))
im = fine_clip.plot(ax=ax, cmap="RdYlBu_r", add_colorbar=False)
cb = plt.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cb.set_label("Mean air temperature, 2000–2024 (°C)")
division.boundary.plot(ax=ax, color="k", linewidth=0.7)
country.boundary.plot(ax=ax, color="k", linewidth=1.5)
ax.set_title("ERA5 mean temperature clipped to Bangladesh\n"
             "(interpolated to 0.01° for display)")
ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
plt.tight_layout(); plt.show()

print(f"mean over Bangladesh : {float(fine_clip.mean()):.2f} C")
print(f"warmest              : {float(fine_clip.max()):.2f} C")
print(f"coolest              : {float(fine_clip.min()):.2f} C")

---
# Part 4 — Zonal Statistics

**Zonal statistics** = a summary of the raster cells inside each polygon.
"What is the mean temperature in Rangamati district?" — that is the question it answers.

In Notebook 2 we used the **point-based** approach (a spatial join),
and one district was left out. Today's **raster-based** approach with
`all_touched=True` covers **every district** — which makes it more reliable.

In [ ]:
# clipping to each district and computing the statistics
rh_mean = (ds["r"].mean("valid_time").rio.write_crs("EPSG:4326")
           .rio.set_spatial_dims(x_dim="longitude", y_dim="latitude"))

recs = []
for _, row in district.iterrows():
    geom = [row.geometry]
    try:
        ct = t_mean.rio.clip(geom, district.crs, all_touched=True, drop=True)
        cr = rh_mean.rio.clip(geom, district.crs, all_touched=True, drop=True)
        recs.append({
            "ADM2_EN": row["ADM2_EN"], "ADM1_EN": row["ADM1_EN"],
            "t_mean": float(ct.mean()), "t_min": float(ct.min()),
            "t_max": float(ct.max()), "t_std": float(ct.std()),
            "rh_mean": float(cr.mean()),
            "n_cells": int(ct.notnull().sum()),
        })
    except Exception as e:
        recs.append({"ADM2_EN": row["ADM2_EN"], "ADM1_EN": row["ADM1_EN"],
                     "t_mean": np.nan, "t_min": np.nan, "t_max": np.nan,
                     "t_std": np.nan, "rh_mean": np.nan, "n_cells": 0})

zonal = pd.DataFrame(recs)
print(f"number of districts       : {len(zonal)}")
print(f"districts with a value    : {zonal['t_mean'].notna().sum()}")
print(f"empty (no cell at all)    : {zonal['t_mean'].isna().sum()}")
print(f"mean cells per district   : {zonal['n_cells'].mean():.1f}")
zonal.head()

In [ ]:
# point-based (NB2) versus raster-based (NB3)
pts = gpd.GeoDataFrame(
    {"t": t_mean.values.ravel()},
    geometry=gpd.points_from_xy(
        *[a.ravel() for a in np.meshgrid(t_mean.longitude.values,
                                         t_mean.latitude.values)][::-1][::-1]),
    crs="EPSG:4326")
LON, LAT = np.meshgrid(t_mean.longitude.values, t_mean.latitude.values)
pts = gpd.GeoDataFrame({"t": t_mean.values.ravel()},
                       geometry=gpd.points_from_xy(LON.ravel(), LAT.ravel()),
                       crs="EPSG:4326")
pj = gpd.sjoin(pts, district, how="inner", predicate="within")
pt_mean = pj.groupby("ADM2_EN")["t"].mean().rename("t_point")

cmp = zonal.merge(pt_mean, on="ADM2_EN", how="left")
cmp["diff"] = (cmp["t_mean"] - cmp["t_point"]).abs()
print("Comparing the methods")
print(f"  point-based found  : {cmp['t_point'].notna().sum()} / {len(cmp)} districts")
print(f"  raster-based found : {cmp['t_mean'].notna().sum()} / {len(cmp)} districts")
print(f"  mean difference    : {cmp['diff'].mean():.3f} C")
print(f"  maximum difference : {cmp['diff'].max():.3f} C")
print("\n-> the all_touched method leaves no district out, so it is the one we recommend.")
print("  Write in the journal Methods: 'Zonal means were computed using")
print("  all_touched rasterization to ensure full coverage of small districts.'")

In [ ]:
# the district results on a map
dz = district.merge(zonal, on=["ADM2_EN", "ADM1_EN"], how="left")

fig, ax = plt.subplots(1, 3, figsize=(16, 5.4))
specs = [("t_mean", "RdYlBu_r", "Mean temperature (°C)", "(a) Mean T"),
         ("rh_mean", "YlGnBu", "Mean relative humidity (%)", "(b) Mean RH"),
         ("t_std", "Purples", "Within-district SD (°C)", "(c) Spatial variability")]
for a, (col, cm, lab, ttl) in zip(ax, specs):
    dz.plot(column=col, ax=a, cmap=cm, legend=True, edgecolor="white", linewidth=0.25,
            legend_kwds={"label": lab, "shrink": 0.72})
    division.boundary.plot(ax=a, color="k", linewidth=0.6)
    a.set_title(ttl); a.set_xlabel("Longitude (°E)"); a.set_ylabel("")
plt.tight_layout(); plt.show()

print("Three warmest:", ", ".join(zonal.nlargest(3, "t_mean")["ADM2_EN"]))
print("Three coolest:", ", ".join(zonal.nsmallest(3, "t_mean")["ADM2_EN"]))

---
# Part 5 — GeoTIFF and Cloud-Optimized GeoTIFF (COG)

## 5.1 GeoTIFF

**GeoTIFF** = an ordinary TIFF image + the geographic information (CRS and transform) written
inside it. It is the most common raster format; QGIS, ArcGIS and everything else can read it.

## 5.2 COG — the modern form

A **COG** is a GeoTIFF arranged in a special way, with two properties —

1. **Tiled** — the image is divided into small tiles (512x512, say)
2. **Overviews** — smaller versions (1/2, 1/4, 1/8 …) stored inside it

**Why is this a revolution?** From a 10 GB COG sitting on the internet you can read only
**the part you need** (a few MB) — without downloading the whole file.
This is the foundation of Part 7.

In [ ]:
# an ordinary GeoTIFF versus a COG
p_tif = OUT/"era5_tmean.tif"
p_cog = OUT/"era5_tmean_cog.tif"

t_fine.rio.to_raster(p_tif)                                     # ordinary
t_fine.rio.to_raster(p_cog, driver="COG", compress="DEFLATE")   # COG

for label, p in [("GeoTIFF (ordinary)", p_tif), ("COG", p_cog)]:
    with rasterio.open(p) as src:
        print(f"{label}")
        print(f"   size       : {src.width} x {src.height}")
        print(f"   block      : {src.block_shapes[0]}   tiled = {src.profile.get('tiled', False)}")
        print(f"   overviews : {src.overviews(1)}")
        print(f"   compress   : {src.profile.get('compress')}")
        print(f"   file       : {p.stat().st_size/1e6:.2f} MB\n")

In [ ]:
# The real magic of a COG — reading a part without reading the whole file
with rasterio.open(p_cog) as src:
    full_px = src.width * src.height

    # (a) a small area — around Dhaka
    row, col = src.index(90.4, 23.8)
    win = Window(col - 50, row - 50, 100, 100)
    sub = src.read(1, window=win)
    print("(a) Windowed read — only the area around Dhaka")
    print(f"     read {sub.size:,} cells  ({100*sub.size/full_px:.2f}% of the whole image)")
    print(f"     mean temperature {np.nanmean(sub):.2f} C")

    # (b) reading from an overview — the whole area, but at lower resolution
    ov = src.read(1, out_shape=(1, src.height//4, src.width//4))
    print("\n(b) Overview read — the whole area, at 1/4 resolution")
    print(f"     read {ov.size:,} cells  ({100*ov.size/full_px:.2f}% of the whole image)")
    print(f"     mean temperature {np.nanmean(ov[ov > -999]):.2f} C")

    print(f"\n-> in both cases only a small part of a {full_px:,}-cell image was read.")
    print("  For a file in the cloud this means downloading a few MB instead of 10 GB!")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.4))
ax[0].imshow(np.where(sub < -900, np.nan, sub), cmap="RdYlBu_r")
ax[0].set_title("(a) Windowed read — Dhaka area, full resolution")
ax[1].imshow(np.where(ov < -900, np.nan, ov), cmap="RdYlBu_r")
ax[1].set_title("(b) Overview read — whole country, 1/4 resolution")
for a in ax: a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.tight_layout(); plt.show()

---
# Part 6 — How to download remote sensing data

You do not have remote sensing data yet. Here we show **exactly how** to get it —
all you need is a free account.

## 6.1 ERA5 (where the data in your hands came from)

**Copernicus Climate Data Store (CDS)** — free, you only have to register.

**Steps:**
1. Open an account -> https://cds.climate.copernicus.eu
2. Log in and take your **API key** (on the Profile page)
3. Put the key in a `~/.cdsapirc` file
4. Run `pip install cdsapi` and then the code below

```python
# -- Downloading ERA5 (run this on your own computer) ---------------
# Inside the ~/.cdsapirc file:
#   url: https://cds.climate.copernicus.eu/api
#   key: <your-API-key>

import cdsapi
c = cdsapi.Client()

c.retrieve(
    "reanalysis-era5-pressure-levels-monthly-means",
    {
        "product_type": "monthly_averaged_reanalysis",
        "variable": ["temperature", "relative_humidity"],
        "pressure_level": "1000",
        "year":  [str(y) for y in range(2000, 2025)],
        "month": [f"{m:02d}" for m in range(1, 13)],
        "time": "00:00",
        "area": [27, 87, 20, 94],        # north, west, south, east
        "format": "netcdf",
    },
    "era5_monthly_bangladesh.nc",       # <- this is exactly how your file was produced
)
```

> **Notice:** `area: [27, 87, 20, 94]` — exactly the bounds of our file!
> And changing `product_type` to `"monthly_averaged_reanalysis_by_hour_of_day"`
> gives you your **second** file (the hourly one).

## 6.2 Sentinel-2 (10 metres, every 5 days, free)

| Source | Address | Advantage |
|------|---------|---------|
| Copernicus Browser | dataspace.copernicus.eu | official, the full archive |
| Microsoft Planetary Computer | planetarycomputer.microsoft.com | **STAC, no download needed** |
| AWS Earth Search | earth-search.aws.element84.com | **STAC, not even registration** |

## 6.3 Landsat 8/9 (30 metres, since 1972)

| Source | Address |
|------|---------|
| USGS EarthExplorer | earthexplorer.usgs.gov |
| Planetary Computer | `collections=["landsat-c2-l2"]` |

```python
# -- Downloading Sentinel-2 from the Copernicus Data Space (real code) ------
# pip install openeo
import openeo
conn = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()
cube = conn.load_collection(
    "SENTINEL2_L2A",
    spatial_extent={"west": 90.2, "south": 23.6, "east": 90.6, "north": 24.0},
    temporal_extent=["2024-01-01", "2024-03-31"],
    bands=["B04", "B08"],
    max_cloud_cover=20,
)
cube.download("sentinel2_dhaka.nc")
```

> **But downloading is not the best way at all —** see the next Part.

---
# Part 7 — STAC: using satellite imagery **without** downloading

## 7.1 The problem

One Sentinel-2 scene over Bangladesh is about **700 MB**. Every scene for one year is
**several hundred GB**. It will not fit on your laptop, and downloading would take days.

But perhaps all you need is **one small area around Dhaka** and **two bands**.
There is no sense in fetching the whole thing.

## 7.2 The solution: STAC + COG

**STAC** (*SpatioTemporal Asset Catalog*) is a **standard catalogue** of satellite imagery —
a common way of asking "what imagery exists, where and when".

| Step | What happens |
|-----|--------|
| 1 | you search the STAC API — area, date, how much cloud |
| 2 | results come back — which images match, and their **addresses (URLs)** |
| 3 | those images sit in the cloud in **COG** format |
| 4 | only **the piece covering your area** is read from the COG |

**The result:** instead of 700 MB you may read perhaps **3 MB**. The job is done in seconds.

> This is the professional method of 2026 — NASA, ESA, Google and Microsoft
> all serve their data this way.

In [ ]:
# -- Internet probe 
import urllib.request
STAC_URL = "https://earth-search.aws.element84.com/v1"   # no registration needed
PC_URL   = "https://planetarycomputer.microsoft.com/api/stac/v1"

NET_OK = False
try:
    urllib.request.urlopen(STAC_URL, timeout=8)
    NET_OK = True
    print("Internet available — the real STAC catalogue will be used.")
except Exception as e:
    print(f"No internet ({type(e).__name__}) — a synthetic sample will be used instead.")
    print("  (all the code below works exactly as written on your own computer)")

### 7.3 Searching STAC — the real code

The cell below will find **genuine Sentinel-2 imagery** on your computer.
We have chosen **a small area around Dhaka**.

In [ ]:
# the area of interest (AOI) — Dhaka and its surroundings
AOI = [90.25, 23.65, 90.55, 23.95]          # west, south, east, north
DATE = "2024-01-01/2024-03-31"               # the dry season = less cloud

items = []
if NET_OK:
    try:
        import pystac_client
        catalog = pystac_client.Client.open(STAC_URL)
        search = catalog.search(
            collections=["sentinel-2-l2a"],
            bbox=AOI,
            datetime=DATE,
            query={"eo:cloud_cover": {"lt": 10}},    # less than 10% cloud
        )
        items = list(search.items())
        print(f"Found {len(items)} Sentinel-2 scenes\n")
        for it in items[:5]:
            print(f"   {it.id}")
            print(f"      date {it.datetime:%Y-%m-%d}  |  "
                  f"cloud {it.properties['eo:cloud_cover']:.1f}%")
    except Exception as e:
        print("Problem searching STAC:", str(e)[:100]); NET_OK = False

if not items:
    print("-> the rest of the work will be shown with a synthetic scene.")

In [ ]:
print("NET_OK      :", NET_OK)
print("scenes found:", len(items))
print("STAC_URL    :", STAC_URL)

### What is inside a STAC Item?

```python
item.id                              # the name of the scene
item.datetime                        # when it was taken
item.geometry                        # the area it covers (GeoJSON)
item.properties["eo:cloud_cover"]    # the percentage of cloud
item.assets                          # the URLs of the bands
item.assets["red"].href              # the direct address of the red band (a COG)
item.assets["nir"].href              # the address of the NIR band
```

Those `href` values are **the addresses of COG files sitting in the cloud**. We will read
only the piece we need from them.

In [ ]:
# -- Reading imagery without downloading it ------------------------------
# two ways are shown:
#   (a) part of one band with rioxarray
#   (b) a whole time series at once with odc.stac (lazily)

red = nir = None

if items:
    try:
        import rioxarray
        it = items[0]
        print("scene:", it.id)
        print("address of the red band:")
        print("  ", it.assets["red"].href[:95], "...")

        # overview_level=2 -> reads the 1/4 resolution version of the image (much faster)
        red = rioxarray.open_rasterio(it.assets["red"].href,
                                      overview_level=2, masked=True).squeeze()
        nir = rioxarray.open_rasterio(it.assets["nir"].href,
                                      overview_level=2, masked=True).squeeze()
        # only our AOI
        box = gpd.GeoDataFrame(geometry=[__import__("shapely").geometry.box(*AOI)],
                               crs="EPSG:4326").to_crs(red.rio.crs)
        red = red.rio.clip(box.geometry, box.crs, drop=True)
        nir = nir.rio.clip(box.geometry, box.crs, drop=True)
        print(f"\nRead — shape {red.shape}, only a few MB of data")
        print("  (the whole scene is ~700 MB — we read only a small part of it)")
    except Exception as e:
        print("Problem reading:", str(e)[:120]); red = nir = None

if red is None:
    # -- Building a synthetic scene (so you can learn without internet) --------
    print("\nBuilding a synthetic Sentinel-2 scene —")
    print("  this is not a real image, but its structure and the arithmetic are identical.")
    rng = np.random.RandomState(42)
    H = W = 400
    yy, xx = np.mgrid[0:H, 0:W]
    river = np.exp(-((xx - 0.5*W - 60*np.sin(yy/55))**2) / (2*14**2))
    urban = (np.exp(-(((xx-120)**2 + (yy-300)**2)/(2*45**2))) +
             np.exp(-(((xx-300)**2 + (yy-120)**2)/(2*35**2))))
    veg = 0.55 + 0.25*np.sin(xx/70)*np.cos(yy/90) + 0.08*rng.randn(H, W)

    nir_a = np.clip(0.42*veg + 0.05, 0.02, 0.6)
    red_a = np.clip(0.09 - 0.05*veg + 0.03*rng.randn(H, W), 0.01, 0.4)
    nir_a = np.where(river > 0.35, 0.03, nir_a); red_a = np.where(river > 0.35, 0.05, red_a)
    nir_a = np.where(urban > 0.40, 0.22, nir_a); red_a = np.where(urban > 0.40, 0.21, red_a)

    xs = np.linspace(AOI[0], AOI[2], W); ys = np.linspace(AOI[3], AOI[1], H)
    red = xr.DataArray(red_a, coords={"y": ys, "x": xs}, dims=("y", "x")
                       ).rio.write_crs("EPSG:4326")
    nir = xr.DataArray(nir_a, coords={"y": ys, "x": xs}, dims=("y", "x")
                       ).rio.write_crs("EPSG:4326")
    print(f"  synthetic scene built — shape {red.shape}")

### 7.4 A whole time series with odc-stac (more powerful still)

To get not one scene but **every scene at once** as an xarray cube —

```python
import odc.stac

cube = odc.stac.load(
    items,                       # the result of the STAC search
    bands=["red", "nir"],
    bbox=AOI,
    resolution=0.0002,           # in degrees (~20 metres)
    chunks={},                   # <- Dask: lazy loading
)
print(cube)        # nothing has been downloaded yet!

ndvi = (cube.nir - cube.red) / (cube.nir + cube.red)
ndvi_mean = ndvi.mean("time").compute()   # <- only now is the needed part read
```

> **Why does `chunks={}` matter?** It turns on **Dask** — the data is read only when it is
> genuinely needed (`.compute()`). So a 100 GB catalogue can be worked with
> on a laptop with 8 GB of RAM.

---
# Part 8 — NDVI: measuring the health of vegetation (band math)

## 8.1 The idea

A green leaf **absorbs red light** (for photosynthesis) but
**reflects near-infrared (NIR) light strongly**. That difference is a measure of the
amount and health of vegetation.

$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}$$

| NDVI | Meaning |
|------|------|
| −1.0 to 0 | water, cloud, snow |
| 0 – 0.2 | bare soil, sand, built-up areas |
| 0.2 – 0.4 | grass, sparse vegetation |
| 0.4 – 0.7 | cropland, moderate forest |
| 0.7 – 1.0 | dense green forest |

**On Sentinel-2:** Red = B04, NIR = B08 (both at 10 metres)
**On Landsat 8/9:** Red = B4, NIR = B5 (30 metres)

In [ ]:
# computing the NDVI — just one line
ndvi = (nir - red) / (nir + red)
ndvi = ndvi.clip(-1, 1)

v = ndvi.values[np.isfinite(ndvi.values)]
print(f"NDVI range : {v.min():.3f}  ->  {v.max():.3f}")
print(f"mean       : {v.mean():.3f}")
print(f"median     : {np.median(v):.3f}")

classes = {
    "water / cloud (< 0)":          (v < 0).sum(),
    "bare soil / built-up (0-0.2)": ((v >= 0) & (v < 0.2)).sum(),
    "sparse vegetation (0.2-0.4)":  ((v >= 0.2) & (v < 0.4)).sum(),
    "crops / moderate forest (0.4-0.7)": ((v >= 0.4) & (v < 0.7)).sum(),
    "dense forest (> 0.7)":         (v >= 0.7).sum(),
}
print("\nDistribution by class:")
for k, c in classes.items():
    print(f"  {k:30s} {c:8,d}  ({100*c/v.size:5.1f}%)")

In [ ]:
# the NDVI map and histogram
from matplotlib.colors import ListedColormap, BoundaryNorm

fig = plt.figure(figsize=(15.5, 5))
gs = fig.add_gridspec(1, 3, width_ratios=[1.1, 1.1, 0.85], wspace=0.55)

# (a) continuous NDVI
a0 = fig.add_subplot(gs[0])
im = a0.imshow(ndvi.values, cmap="RdYlGn", vmin=-0.3, vmax=0.9,
               extent=[AOI[0], AOI[2], AOI[1], AOI[3]])
plt.colorbar(im, ax=a0, shrink=0.8, label="NDVI")
a0.set_title("(a) NDVI"); a0.set_xlabel("Longitude (°E)"); a0.set_ylabel("Latitude (°N)")

# (b) classified
bounds = [-1, 0, 0.2, 0.4, 0.7, 1.0]
cols = ["#2c7bb6", "#d7b5a6", "#fdae61", "#a6d96a", "#1a9641"]
cmap = ListedColormap(cols); norm = BoundaryNorm(bounds, cmap.N)
a1 = fig.add_subplot(gs[1])
im2 = a1.imshow(ndvi.values, cmap=cmap, norm=norm,
                extent=[AOI[0], AOI[2], AOI[1], AOI[3]])
cb = plt.colorbar(im2, ax=a1, shrink=0.8, ticks=[-0.5, 0.1, 0.3, 0.55, 0.85])
cb.ax.set_yticklabels(["Water", "Bare/Urban", "Sparse", "Crop/Forest", "Dense"])
a1.set_title("(b) NDVI classes"); a1.set_xlabel("Longitude (°E)"); a1.set_ylabel("")

# (c) histogram
a2 = fig.add_subplot(gs[2])
a2.hist(v, bins=60, color=C_COOL, edgecolor="white", linewidth=0.4)
for b in bounds[1:-1]:
    a2.axvline(b, color=C_WARM, ls="--", lw=1)
a2.set_xlabel("NDVI"); a2.set_ylabel("Pixel count")
a2.set_title("(c) Distribution")

plt.suptitle("NDVI over the Dhaka study area", fontweight="bold", y=1.02)
plt.savefig(OUT/"fig_nb3_ndvi.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: outputs/fig_nb3_ndvi.png (300 dpi)")

In [ ]:
# saving the NDVI as a GeoTIFF/COG
ndvi_out = ndvi.rio.write_crs(red.rio.crs)
ndvi_out.rio.to_raster(OUT/"ndvi_dhaka_cog.tif", driver="COG", compress="DEFLATE")
print(f"NDVI saved: outputs/ndvi_dhaka_cog.tif "
      f"({(OUT/'ndvi_dhaka_cog.tif').stat().st_size/1e6:.2f} MB)")

# saving the district climate results (Notebook 4 will use them)
dz.to_parquet(OUT/"district_zonal_climate.parquet")
zonal.to_csv(OUT/"table2_district_zonal.csv", index=False)
t_fine.rio.to_raster(OUT/"era5_tmean_fine_cog.tif", driver="COG", compress="DEFLATE")

print("\nFiles saved:")
for f in ["district_zonal_climate.parquet", "table2_district_zonal.csv",
          "era5_tmean_fine_cog.tif", "ndvi_dhaka_cog.tif", "era5_tmean_cog.tif"]:
    if (OUT/f).exists():
        print(f"  {f:34s} {(OUT/f).stat().st_size/1e6:7.2f} MB")

---
# Home Work — Class 10

**1.** Take the 25-year mean of `ds["r"]` (humidity) and clip it to the shape of Bangladesh
(`all_touched=True`). Draw the map and print the national mean humidity.

**2.** Build a raster of the **difference** between the monsoon (JJAS) and dry (DJF) mean
temperatures, clip it to Bangladesh and draw it. Where is the difference largest?

**3.** Reproject `t_fine` to **EPSG:3106** and save it as a COG.
Then open it with `rasterio` and print the overviews and the block size.

**4.** Compute zonal statistics for each **division** —
the mean, minimum and maximum temperature. Show them in a table.

**5.** *(If you have internet)* change the date in the STAC search to the **2023 monsoon**
(June–September), and raise the cloud limit to 20%. How many scenes do you find?
Is the NDVI higher in the monsoon than in the dry season?

---
# Class 10 — Summary

| What we learned | Key command |
|-----------|-------------|
| adding a CRS | `.rio.write_crs("EPSG:4326")` |
| naming the axes | `.rio.set_spatial_dims(x_dim=, y_dim=)` |
| inspecting it | `.rio.resolution()` `.rio.bounds()` `.rio.transform()` |
| changing the CRS | `.rio.reproject("EPSG:3106")` |
| changing the resolution | `.interp(...)` |
| cutting to a shape | `.rio.clip(geom, crs, all_touched=True)` |
| writing a GeoTIFF | `.rio.to_raster("x.tif")` |
| writing a COG | `.rio.to_raster("x.tif", driver="COG")` |
| reading a part | `src.read(1, window=Window(...))` |
| searching STAC | `pystac_client.Client.open(...).search(...)` |
| reading from afar | `rioxarray.open_rasterio(href, overview_level=2)` |
| a lazy cube | `odc.stac.load(items, chunks={})` |
| NDVI | `(nir - red) / (nir + red)` |

---

### The most common mistakes

1. **Clipping without a CRS** — without `write_crs` first, `.rio` will not work.
2. **Bilinear resampling on categorical data** — for land cover it is always `nearest`.
3. **Forgetting `all_touched`** — on a coarse grid, small districts drop out.
4. **Claiming more precision after interpolating** — the native resolution does not change.
5. **Downloading a whole scene** — use STAC + COG and read only the part you need.
6. **The wrong bands in NDVI** — Sentinel-2: B08 and B04; Landsat: B5 and B4.

### What comes next

**Class 11 — Advanced Analysis & Geovisualization**: trend analysis
(Mann-Kendall, Sen's slope), spatial autocorrelation (Moran's I, LISA),
clustering, interactive maps, and **journal-ready figures**.

---

**Md Wabidur Rahman** — Researcher, Research and Integrated Thought (RIT)
*Geospatial Data Analysis and Visualization · Class 10 — Raster Data & Remote Sensing*